### 🧠 The Final Production Notebook

Since you want the final, clean code without the iterations, here is the consolidated version. You can create a fresh notebook called **`SCD_Production_Model.ipynb`** and paste this in. It runs sequentially, handles all the NaNs, applies WoE correctly, and trains the three models.

```python
# %% [markdown]
# # SCD IFRS 9 Production Pipeline
# This notebook contains the clean, end-to-end pipeline for the Synthetic Credit Data (SCD) project.

# %% [markdown]

In [1]:
# ### 1. Load Data & Create 60-Month Panel
# %%
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

ROOT_DIR = r"C:\CreditRiskCourse\Project_6_SCD_IFRS9_Staging"
DATA_DIR = os.path.join(ROOT_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(ROOT_DIR, "data", "processed")

# Load
customers = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))
accounts = pd.read_csv(os.path.join(DATA_DIR, "accounts.csv"), parse_dates=['opening_date'])
payments = pd.read_csv(os.path.join(DATA_DIR, "payments.csv"), parse_dates=['payment_date'])
debt_reviews = pd.read_csv(os.path.join(DATA_DIR, "debt_reviews.csv"), parse_dates=['review_date'])
restructures = pd.read_csv(os.path.join(DATA_DIR, "restructures.csv"), parse_dates=['restructure_date'])
scores = pd.read_csv(os.path.join(DATA_DIR, "credit_scores.csv"), parse_dates=['score_date'])

# Clean payments
payments = payments[payments['amount_paid'] >= 0]
payments = payments.drop_duplicates(subset=['payment_id'])

# Create Panel
months = pd.period_range(start='2020-01', end='2024-12', freq='M')
accounts_list = accounts['account_id'].unique()
panel = pd.MultiIndex.from_product([accounts_list, months], names=['account_id', 'month']).to_frame(index=False)
panel['month_date'] = panel['month'].dt.to_timestamp()

# Merge Static Data
panel = panel.merge(accounts[['account_id', 'customer_id', 'product_id', 'current_balance', 'interest_rate']], on='account_id', how='left')

# %% [markdown]
# ### 2. Calculate Monthly IFRS 9 Stages
# %%
# 1. Missed Payments (Rolling 12m)
payments['payment_month'] = payments['payment_date'].dt.to_period('M')
payments['shortfall'] = payments['amount_due'] - payments['amount_paid']
payments['is_missed'] = (payments['shortfall'] > (payments['amount_due'] * 0.1)).astype(int)
monthly_pay = payments.groupby(['account_id', 'payment_month'])['is_missed'].sum().reset_index()

panel = panel.merge(monthly_pay, left_on=['account_id', 'month'], right_on=['account_id', 'payment_month'], how='left').fillna({'is_missed': 0})
panel = panel.drop(columns=['payment_month']).sort_values(['account_id', 'month_date'])
panel['missed_12m'] = panel.groupby('account_id')['is_missed'].rolling(12, min_periods=1).sum().reset_index(drop=True)

# 2. Debt Review
debt_reviews['review_month'] = debt_reviews['review_date'].dt.to_period('M')
dr_active = debt_reviews[debt_reviews['review_outcome'].isin(['Plan Initiated', 'Completed'])][['customer_id', 'review_month']]
panel = panel.merge(dr_active, on='customer_id', how='left')
panel['in_debt_review'] = (panel['review_month'] <= panel['month']).astype(int)
panel['in_debt_review'] = panel.groupby(['account_id', 'month'])['in_debt_review'].transform('max')
panel = panel.drop(columns=['review_month'])

# 3. Restructures
restructures['rest_month'] = restructures['restructure_date'].dt.to_period('M')
rest_active = restructures[restructures['status'].isin(['Active', 'Broken'])][['account_id', 'rest_month', 'status']]
panel = panel.merge(rest_active, on='account_id', how='left')
panel['restructure_status'] = np.where(panel['rest_month'] <= panel['month'], panel['status'], None)
panel = panel.drop(columns=['rest_month', 'status'])

# 4. Credit Score
scores['score_month'] = scores['score_date'].dt.to_period('M')
latest_scores = scores.sort_values('score_date').groupby(['customer_id', 'score_month']).last().reset_index()
panel = panel.merge(latest_scores[['customer_id', 'score_month', 'fico_score']], left_on=['customer_id', 'month'], right_on=['customer_id', 'score_month'], how='left')
panel['credit_score'] = panel.groupby('account_id')['fico_score'].ffill().bfill()
panel['credit_score'] = panel['credit_score'].fillna(panel['credit_score'].median())
panel = panel.drop(columns=['score_month', 'fico_score'])

# 5. Staging
def assign_stage(row):
    if row['in_debt_review'] == 1 or row['restructure_status'] == 'Broken' or row['missed_12m'] >= 3:
        return 'Stage 3'
    elif row['restructure_status'] == 'Active' or row['missed_12m'] >= 2 or row['credit_score'] < 580:
        return 'Stage 2'
    else:
        return 'Stage 1'

panel['IFRS9_Stage'] = panel.apply(assign_stage, axis=1)

# %% [markdown]
# ### 3. Train Product-Specific Models & Calculate ECL
# %%
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Setup features
features = ['current_balance', 'interest_rate', 'missed_12m', 'credit_score']
train_mask = panel['month'] <= '2023-12'
test_mask = panel['month'] >= '2024-01'
panel['is_default'] = (panel['IFRS9_Stage'] == 'Stage 3').astype(int)

# Define Dynamic LGD
def calculate_dynamic_lgd(row):
    if row['product_id'] == 'HL': base_lgd = 0.20
    elif row['product_id'] == 'AL': base_lgd = 0.30
    else: base_lgd = 0.45
    if row['in_debt_review'] == 1: base_lgd += 0.15
    elif row['restructure_status'] == 'Active': base_lgd += 0.10
    elif row['restructure_status'] == 'Broken': base_lgd += 0.20
    return min(base_lgd, 1.0)

panel['LGD'] = panel.apply(calculate_dynamic_lgd, axis=1)
panel['EAD'] = panel['current_balance']
panel['PD_Predicted'] = 0.0

# WoE Function
def calculate_woe_iv(df, feature, target):
    df_temp = df[[feature, target]].copy()
    if feature == 'missed_12m':
        df_temp['bin'] = pd.cut(df_temp[feature], bins=[-1, 0, 1, 2, 100], labels=['0', '1', '2', '3+']).astype(str)
    else:
        df_temp['bin'] = pd.qcut(df_temp[feature], q=5, duplicates='drop').astype(str)
    grouped = df_temp.groupby('bin', observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['Total', 'Bad']
    grouped['Good'] = grouped['Total'] - grouped['Bad']
    grouped['Bad'] = grouped['Bad'].replace(0, 0.5)
    grouped['Good'] = grouped['Good'].replace(0, 0.5)
    grouped['Dist_Good'] = grouped['Good'] / grouped['Good'].sum()
    grouped['Dist_Bad'] = grouped['Bad'] / grouped['Bad'].sum()
    grouped['WoE'] = np.log(grouped['Dist_Good'] / grouped['Dist_Bad'])
    return grouped

# Loop through products
models = {}
for prod in ['HL', 'AL', 'PL']:
    df_train = panel[(panel['product_id'] == prod) & train_mask].copy()
    df_test = panel[(panel['product_id'] == prod) & test_mask].copy()
    
    for feat in features:
        woe_table = calculate_woe_iv(df_train, feat, 'is_default')
        bin_mapping = woe_table['WoE'].to_dict()
        if feat == 'missed_12m':
            df_train[feat] = pd.cut(df_train[feat], bins=[-1, 0, 1, 2, 100], labels=['0', '1', '2', '3+']).astype(str).map(bin_mapping).fillna(0)
            df_test[feat] = pd.cut(df_test[feat], bins=[-1, 0, 1, 2, 100], labels=['0', '1', '2', '3+']).astype(str).map(bin_mapping).fillna(0)
        else:
            df_train[feat] = pd.qcut(df_train[feat], q=5, duplicates='drop').astype(str).map(bin_mapping).fillna(0)
            df_test[feat] = pd.qcut(df_test[feat], q=5, duplicates='drop').astype(str).map(bin_mapping).fillna(0)
            
    X_train, y_train = df_train[features], df_train['is_default']
    X_test, y_test = df_test[features], df_test['is_default']
    
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    y_pred = lr.predict_proba(X_test)[:, 1]
    models[prod] = lr
    
    # Assign Predictions back
    panel.loc[df_test.index, 'PD_Predicted'] = y_pred
    print(f"{prod} Model Trained. AUC: {roc_auc_score(y_test, y_pred):.4f}")

# For historical data, proxy PDs
panel.loc[train_mask & (panel['IFRS9_Stage'] == 'Stage 1'), 'PD_Predicted'] = 0.02
panel.loc[train_mask & (panel['IFRS9_Stage'] == 'Stage 2'), 'PD_Predicted'] = 0.15
panel.loc[train_mask & (panel['IFRS9_Stage'] == 'Stage 3'), 'PD_Predicted'] = 1.00

# Calculate ECL
panel['ECL'] = panel['PD_Predicted'] * panel['LGD'] * panel['EAD']

# Save
panel.to_csv(os.path.join(PROCESSED_DIR, "SCD_IFRS9_Final.csv"), index=False)
print("✅ Production pipeline complete. Data saved for AI FDE.")

HL Model Trained. AUC: 0.7087
AL Model Trained. AUC: 0.6881
PL Model Trained. AUC: 0.6804
✅ Production pipeline complete. Data saved for AI FDE.
